# 🧠 Approach 2: Deep Learning Transfer Learning (MobileNetV2)

**Objective:** Implement a lightweight, edge-compatible Convolutional Neural Network (MobileNetV2) pretrained on ImageNet, customize the classification head, handle class imbalance via weighted cross-entropy, and train with learning rate scheduling and early stopping.


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from src.dataset import prepare_dataset_splits, get_dataloaders
from src.dl_model import ProduceClassifierCNN, train_dl_model, evaluate_dl_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")


## 1. Load DataLoaders with Data Augmentation
Applies random rotation, flips, color jitter, and ImageNet standardization.


In [ ]:
splits = prepare_dataset_splits('.')
classes = splits['classes']
train_loader, val_loader, test_loader = get_dataloaders(splits, batch_size=32, img_size=224)
print(f"Classes: {classes}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


## 2. Build MobileNetV2 Architecture & Edge Head
Custom classification head with Dropout, Linear(in_features, 256), BatchNorm1d, ReLU, Dropout, Linear(256, 4).


In [ ]:
model = ProduceClassifierCNN(num_classes=len(classes), architecture='mobilenet_v2', pretrained=True)
model.freeze_backbone()
print(model.backbone.classifier)


## 3. Train MobileNetV2 with Class Weights and Early Stopping


In [ ]:
history = train_dl_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=12,
    lr=1e-3,
    class_weights=splits['class_weights'],
    save_path='models/mobilenetv2_best.pth',
    patience=4
)


## 4. Test Set Evaluation & Latency Benchmarking


In [ ]:
eval_results = evaluate_dl_model(model, test_loader, device=device, class_names=classes)
print(f"Test Accuracy:  {eval_results['accuracy']*100:.2f}%")
print(f"Macro F1-Score: {eval_results['macro_f1']:.4f}")
print(f"Average CPU Latency: {eval_results['avg_latency_ms']:.2f} ms/image")

import seaborn as sns
plt.figure(figsize=(6, 5))
sns.heatmap(eval_results['confusion_matrix'], annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('MobileNetV2 Test Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.show()
